<a href="https://colab.research.google.com/github/AyoubElKharraf/NeRF_LRM__From_2D_to_3D/blob/main/03_nerf_lego_30k_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Pipeline NeRF Haute Fidélité (30 000 itérations)
**Projet Master M2I — De la 2D à la 3D : NeRF, pixelNeRF & LRM**

Ce notebook permet d'exécuter le pipeline complet de A à Z sans encombre :
1. **Étape 1 :** Vérification de l'environnement GPU (CUDA)
2. **Étape 2 :** Montage Google Drive (pour sauvegarder automatiquement vos modèles)
3. **Étape 3 :** Installation de Nerfstudio & dépendances
4. **Étape 4 :** Téléchargement automatique du dataset synthétique officiel Lego
5. **Étape 5 :** Entraînement complet du modèle Nerfacto jusqu'à 30 000 itérations
6. **Étape 6 :** Récupération automatique du dossier de sortie
7. **Étape 7 :** Évaluation quantitative du modèle (PSNR, SSIM, LPIPS)
8. **Étape 8 :** Génération du rendu vidéo Haute Définition (sans flou, downscale=1)
9. **Étape 9 :** Visualisation de la vidéo directement dans Colab
10. **Étape 10 :** Sauvegarde complète sur Google Drive

### Étape 1 : Vérification de l'accélérateur GPU (CUDA)

In [ ]:
import torch
print("=== 1. VÉRIFICATION GPU ===")
print(f"Version PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU actif : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ VRAM totale : {vram:.2f} Go")
else:
    raise SystemError("❌ Aucun GPU détecté. Allez dans Exécution -> Modifier le type d'exécution -> GPU T4.")

=== 1. VÉRIFICATION GPU ===
Version PyTorch : 2.14.1+cu130
✅ GPU actif : Tesla T4
✅ VRAM totale : 14.56 Go


### Étape 2 : Montage de Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nerf_lego_30000_final

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Étape 3 : Installation de Nerfstudio & HuggingFace Hub

In [ ]:
!pip install --upgrade pip
!pip install nerfstudio huggingface_hub
print("✅ Nerfstudio et dépendances installés !")

✅ Nerfstudio et dépendances installés !


### Étape 4 : Téléchargement et extraction du Dataset Lego

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile
from pathlib import Path

print("Téléchargement du dataset officiel Lego...")
archive = hf_hub_download(
    repo_id="nerfbaselines/nerfbaselines-data",
    repo_type="dataset",
    filename="blender/lego.zip",
    local_dir="/content/data"
)

output_dir = Path("/content/data/blender")
output_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive) as zip_file:
    zip_file.extractall(output_dir)

print("✅ Dataset extrait avec succès dans :", output_dir / "lego")

Téléchargement du dataset officiel Lego...


✅ Dataset extrait avec succès dans : /content/data/blender/lego


### Étape 5 : Entraînement complet de Nerfacto jusqu'à 30 000 itérations
*(L'option `--vis tensorboard` désactive le viewer web interactif pour afficher la progression directement dans Colab sans aucun blocage !)*

In [ ]:
!ns-train nerfacto \
    --vis tensorboard \
    --max-num-iterations 30000 \
    blender-data \
    --data /content/data/blender/lego

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
25800 (86.00%)      303.534 ms           21 m, 14 s           14.03 K                                    
25810 (86.03%)      296.800 ms           20 m, 43 s           14.35 K                                    
25820 (86.07%)      306.553 ms           21 m, 21 s           14.03 K                                    
25830 (86.10%)      299.288 ms           20 m, 48 s           14.24 K                                    
25840 (86.13%)      301.455 ms           20 m, 54 s           14.12 K                                    
25850 (86.17%)      309.917 ms           21 m, 26 s           13.84 K                                    
Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec     Test Rays / Sec       
-------------------------------------------------------------------------------------------------------- 
25770 (85.90%)      296.751 ms           20 m, 55 s           14.35 K     

### Étape 6 : Récupération automatique du chemin du modèle entraîné

In [ ]:
import glob
import os

configs = sorted(glob.glob('/content/outputs/unnamed/nerfacto/*/config.yml'), key=os.path.getmtime)
if not configs:
    raise FileNotFoundError("Aucun dossier de sortie trouvé dans /content/outputs/ !")

latest_config = configs[-1]
print(f"✅ Dernier modèle trouvé : {latest_config}")

✅ Dernier modèle trouvé : /content/outputs/unnamed/nerfacto/2026-11-25_091818/config.yml


### Étape 7 : Évaluation quantitative (Calcul du PSNR, SSIM, LPIPS)

In [ ]:
import json
!mkdir -p /content/outputs/metrics
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-eval --load-config {latest_config} --output-path /content/outputs/metrics/metrics_30000.json

with open('/content/outputs/metrics/metrics_30000.json', 'r') as f:
    metrics = json.load(f)

print("=== RÉSULTATS QUANTITATIFS (30 000 itérations) ===")
print(json.dumps(metrics.get('results', metrics), indent=4))

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1795611292.108081   55654 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/

### Étape 8 : Génération de la vidéo HD nette et cristalline (downscale-factor=1)

In [ ]:
!mkdir -p /content/outputs/renders

!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render interpolate \
    --load-config {latest_config} \
    --output-path /content/outputs/renders/lego_30000_hd_cristal.mp4 \
    --downscale-factor 1 \
    --rendered-output-names rgb

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1795615852.087095   74409 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/

### Étape 9 : Visualisation du résultat vidéo dans Colab

In [ ]:
from IPython.display import Video, display
display(Video('/content/outputs/renders/lego_30000_spiral_cristal.mp4', embed=True, width=640))

### Étape 10 : Sauvegarde complète sur Google Drive

In [ ]:
import os
run_dir = os.path.dirname(latest_config)

!cp -r {run_dir} /content/drive/MyDrive/nerf_lego_30000_final/
!cp /content/outputs/metrics/metrics_30000.json /content/drive/MyDrive/nerf_lego_30000_final/
!cp /content/outputs/renders/lego_30000_spiral_cristal.mp4 /content/drive/MyDrive/nerf_lego_30000_final/

print("🎉 Sauvegarde terminée avec succès dans : MyDrive/nerf_lego_30000_final/")

cp: cannot stat '/content/outputs/renders/lego_30000_spiral_cristal.mp4': No such file or directory
🎉 Sauvegarde terminée avec succès dans : MyDrive/nerf_lego_30000_final/
